# Feature Attribution

Which input tokens are responsible for driving specific internal features?
This notebook explores fine-grained attribution tools that connect input
tokens to internal activations.

This notebook covers the `irtk.feature_attribution` module.

## Why This Matters

Feature attribution decomposes model predictions into contributions from individual tokens, neurons, and directions. This bridges the gap between input-level explanations (which tokens matter?) and mechanistic understanding (which internal components produce the prediction?).

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

import irtk
from irtk import feature_attribution

model = irtk.HookedTransformer.from_pretrained("gpt2")
tokenizer = model.tokenizer
print(f"Model: {model.cfg.n_layers} layers, d_mlp={model.cfg.d_mlp}")

## 1. Token-to-Neuron Attribution

Which input tokens are most responsible for activating a specific MLP neuron?

In [ ]:
prompt = "The Eiffel Tower is located in Paris, France"
tokens = model.to_tokens(prompt)
token_strs = [tokenizer.decode([int(t)]) for t in tokens]
print(f"Tokens: {token_strs}")

# Find an interesting neuron first
_, cache = model.run_with_cache(tokens)
mlp_acts = np.array(cache["blocks.8.mlp.hook_post"][-1])  # last position
top_neuron = int(np.argmax(np.abs(mlp_acts)))
print(f"Most active neuron at L8 (last pos): {top_neuron} (act={mlp_acts[top_neuron]:.4f})")

# Attribute to input tokens
attr = feature_attribution.token_to_neuron_attribution(model, tokens, layer=8, neuron=top_neuron)
print(f"\nToken attributions for L8 N{top_neuron}:")
for i, (tok, a) in enumerate(zip(token_strs, attr)):
    bar = '█' * int(a / max(attr) * 20) if max(attr) > 0 else ''
    print(f"  [{i:2d}] {tok:>10s}: {a:.4f} {bar}")

In [ ]:
# Visualize attributions for multiple neurons
fig, axes = plt.subplots(3, 1, figsize=(12, 10))

for ax_idx, neuron in enumerate(np.argsort(np.abs(mlp_acts))[::-1][:3]):
    attr = feature_attribution.token_to_neuron_attribution(
        model, tokens, layer=8, neuron=int(neuron)
    )
    axes[ax_idx].bar(range(len(attr)), attr)
    axes[ax_idx].set_xticks(range(len(token_strs)))
    axes[ax_idx].set_xticklabels(token_strs, rotation=45, ha="right")
    axes[ax_idx].set_ylabel("Attribution")
    axes[ax_idx].set_title(f"L8 Neuron {neuron} (act={mlp_acts[neuron]:.3f})")

plt.tight_layout()
plt.show()

## 2. Token-to-Direction Attribution

Which tokens contribute to a specific direction in activation space?
Useful for understanding concept directions or steering vectors.

In [ ]:
# Use the unembedding direction of " Paris" as our target direction
paris_token = tokenizer.encode(" Paris")[0]
direction = np.array(model.unembed.W_U[:, paris_token])  # [d_model]

hook = "blocks.10.hook_resid_post"
attr = feature_attribution.token_to_direction_attribution(
    model, tokens, hook, direction
)

print(f"Attribution for ' Paris' direction at L10:")
for i, (tok, a) in enumerate(zip(token_strs, attr)):
    bar = '█' * int(a / max(attr) * 20) if max(attr) > 0 else ''
    print(f"  [{i:2d}] {tok:>10s}: {a:.4f} {bar}")

## 3. Decompose Logit by Token

Break down a prediction into per-input-token contributions.
This tells us which tokens are most responsible for a specific prediction.

In [ ]:
prompt = "The capital of France is"
tokens = model.to_tokens(prompt)
token_strs = [tokenizer.decode([int(t)]) for t in tokens]

# What does the model predict?
logits = model(tokens)
top_pred = int(jnp.argmax(logits[-1]))
print(f"Prompt: {prompt!r}")
print(f"Top prediction: {tokenizer.decode([top_pred])!r}")

# Decompose the target logit
target = tokenizer.encode(" Paris")[0]
result = feature_attribution.decompose_logit_by_token(model, tokens, target_token=target)

print(f"\nBaseline logit for ' Paris': {result['baseline_logit']:.4f}")
print(f"Sum of attributions: {result['total_attribution']:.4f}")
print(f"\nPer-token contributions:")
for i, (tok, a) in enumerate(zip(token_strs, result["attributions"])):
    bar = '█' * int(abs(a) / max(abs(result['attributions'])) * 20) if max(abs(result['attributions'])) > 0 else ''
    sign = '+' if a > 0 else '-'
    print(f"  [{i:2d}] {tok:>10s}: {a:+.4f} {sign}{bar}")

In [ ]:
# Compare decomposition for different target tokens
targets = [
    (" Paris", tokenizer.encode(" Paris")[0]),
    (" London", tokenizer.encode(" London")[0]),
    (" Berlin", tokenizer.encode(" Berlin")[0]),
]

fig, axes = plt.subplots(1, len(targets), figsize=(5 * len(targets), 5))
for ax, (name, tid) in zip(axes, targets):
    r = feature_attribution.decompose_logit_by_token(model, tokens, target_token=tid)
    colors = ['green' if a > 0 else 'red' for a in r['attributions']]
    ax.barh(range(len(token_strs)), r['attributions'], color=colors)
    ax.set_yticks(range(len(token_strs)))
    ax.set_yticklabels(token_strs)
    ax.set_xlabel("Attribution")
    ax.set_title(f"Logit decomposition for '{name}'")
    ax.axvline(0, color='black', linewidth=0.5)

plt.tight_layout()
plt.show()

## 4. Feature Importance Ranking

Which dimensions of the residual stream are most active at a given hook point?

In [ ]:
prompt = "The quick brown fox jumps over the lazy dog"
tokens = model.to_tokens(prompt)

ranking = feature_attribution.feature_importance_ranking(
    model, tokens, "blocks.11.hook_resid_post", k=15
)

print("Top 15 most active dimensions at L11 residual (last position):")
for idx, val, mag in zip(
    ranking["top_indices"], ranking["top_values"], ranking["top_magnitudes"]
):
    print(f"  dim {idx:4d}: value={val:+.4f}  magnitude={mag:.4f}")

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(range(len(ranking["full_activations"])), np.abs(ranking["full_activations"]))
ax.set_xlabel("Dimension")
ax.set_ylabel("Absolute Activation")
ax.set_title("Residual Stream Activation Magnitudes (L11, last position)")
plt.tight_layout()
plt.show()

## 5. Cross-Layer Attribution

How do specific dimensions at an earlier layer contribute to a direction
at a later layer? This reveals the causal structure of representations.

In [ ]:
prompt = "The Eiffel Tower is in"
tokens = model.to_tokens(prompt)

# Target: "Paris" direction at late layer
paris_dir = np.array(model.unembed.W_U[:, tokenizer.encode(" Paris")[0]])

result = feature_attribution.cross_layer_attribution(
    model, tokens,
    source_hook="blocks.6.hook_resid_post",
    target_hook="blocks.11.hook_resid_post",
    target_direction=paris_dir,
    n_dims=15,
)

print(f"Baseline projection onto 'Paris' direction: {result['baseline_projection']:.4f}")
print(f"\nTop source dimensions at L6 and their effect on L11's 'Paris' direction:")
for dim, attr in zip(result["source_dims"], result["attributions"]):
    bar = '█' * int(abs(attr) / max(abs(result['attributions'])) * 15) if max(abs(result['attributions'])) > 0 else ''
    print(f"  dim {dim:4d}: attribution={attr:+.4f} {bar}")

fig, ax = plt.subplots(figsize=(10, 4))
colors = ['green' if a > 0 else 'red' for a in result['attributions']]
ax.bar(range(len(result['source_dims'])), result['attributions'], color=colors)
ax.set_xticks(range(len(result['source_dims'])))
ax.set_xticklabels([f"d{d}" for d in result['source_dims']], rotation=45)
ax.set_ylabel("Attribution")
ax.set_title("L6 → L11 Cross-Layer Attribution for 'Paris' Direction")
ax.axhline(0, color='black', linewidth=0.5)
plt.tight_layout()
plt.show()

## Summary

| Function | Purpose |
|----------|--------|
| `token_to_neuron_attribution()` | Which input tokens drive a specific neuron's activation |
| `token_to_direction_attribution()` | Which tokens contribute to a specific direction |
| `decompose_logit_by_token()` | Per-input-token contribution to a logit |
| `feature_importance_ranking()` | Rank the most important features at a hook point |
| `cross_layer_attribution()` | How activations at one layer contribute to a direction at another |